# 第 7 天 · 周复盘 + 缓冲

> 📓 **这是第 7 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/7)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q pydantic tiktoken
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
import os, js
from pyodide.http import pyfetch
os.environ['TIKTOKEN_CACHE_DIR'] = '/tmp/tiktoken'; os.makedirs('/tmp/tiktoken', exist_ok=True)
if not os.path.exists('/tmp/tiktoken/9b5ad71b2ce5302211f9c61530b329a4922fc6a4'):
    # cl100k_base 编码文件改从本站下载（免跨域、国内可访问）
    _r = await pyfetch(str(js.location.href).split('/lite/')[0] + '/pyodide/cl100k_base.tiktoken')
    with open('/tmp/tiktoken/9b5ad71b2ce5302211f9c61530b329a4922fc6a4', 'wb') as f:
        f.write(await _r.bytes())
print("✓ 环境就绪，可以往下运行了")

In [ ]:
"""第 7 天 · 不需要 Key：把一周所学串成一个 ChatSession
用到：dataclass(第2天) + pydantic(第3天) + messages 三角色(第4天)
     + tiktoken 计费(第5天) + async 流式 & 多轮记忆(第6天)
新东西：按 token 预算裁剪历史（比"只留 N 轮"更精确）"""
import asyncio
from dataclasses import dataclass, field
from typing import Literal

import tiktoken
from pydantic import BaseModel, ValidationError

ENC = tiktoken.get_encoding("cl100k_base")
PRICE_IN_PER_M = 2.0   # 输入单价：元 / 百万 token（示例值，按你用的模型改）


class Message(BaseModel):                       # ≈ 带 Bean Validation 的 DTO
    role: Literal["system", "user", "assistant"]
    content: str


def count_tokens(messages: list[Message]) -> int:
    # 每条消息约多 4 个 token 的格式开销，粗略估算够用
    return sum(len(ENC.encode(m.content)) + 4 for m in messages)


async def fake_llm(messages: list[Message]):
    """假模型：async 生成器，模拟 stream=True 的 chunk"""
    users = [m.content for m in messages if m.role == "user"]
    reply = f"（我收到 {len(messages)} 条消息，最早记得的一句：{users[0]}）"
    for i in range(0, len(reply), 4):
        await asyncio.sleep(0.03)
        yield reply[i:i + 4]


@dataclass                                     # ≈ Lombok @Data
class ChatSession:
    system: str
    budget: int = 120                          # 发出去的历史最多多少 token
    history: list[Message] = field(default_factory=list)

    def window(self) -> list[Message]:
        """system 永远保留；从最新往前加，超出预算就停（像 LRU 的反方向）"""
        sys_msg = Message(role="system", content=self.system)
        picked: list[Message] = []
        used = count_tokens([sys_msg])
        for m in reversed(self.history):
            cost = count_tokens([m])
            if used + cost > self.budget:
                break
            picked.insert(0, m)
            used += cost
        return [sys_msg] + picked

    async def ask(self, question: str) -> str:
        self.history.append(Message(role="user", content=question))
        sent = self.window()
        tokens = count_tokens(sent)
        print(f"你：{question}")
        print(f"   [发出 {len(sent)} 条 / {tokens} token ≈ ¥{tokens * PRICE_IN_PER_M / 1e6:.6f}]")
        print("助手：", end="", flush=True)
        parts = []
        async for chunk in fake_llm(sent):
            print(chunk, end="", flush=True)
            parts.append(chunk)
        print("\n")
        answer = "".join(parts)
        self.history.append(Message(role="assistant", content=answer))
        return answer


async def main() -> None:
    s = ChatSession(system="你是一个助手")
    for q in ["我叫 Jiawei", "我是 Java 程序员", "我在学 Agent 开发", "我叫什么？"]:
        await s.ask(q)
    print(f"本地完整历史 {len(s.history)} 条，但最后一次只发出去了窗口内的部分。")

    try:                                       # pydantic 帮你挡住非法角色
        Message(role="boss", content="hi")
    except ValidationError as e:
        print("\n非法消息被拦下：", e.errors()[0]["msg"])


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await

In [ ]:
"""第 7 天 · 找 Bug：这段代码有 3 个 bug（都是前 6 天讲过的坑），改到最后打印 ✅ 为止"""
import asyncio
import time


async def fake_stream(messages: list[dict]):
    """模拟真实 API：最后一个 chunk 的 content 是 None"""
    n = sum(1 for m in messages if m["role"] == "user")
    for piece in ["收到", f"第 {n} 个", "问题", None]:
        time.sleep(0.2)                  # 模拟网络等待
        yield piece


async def chat(messages: list[dict], question: str) -> str:
    messages.append({"role": "user", "content": question})
    parts = []
    async for content in fake_stream(messages):
        parts.append(content)
    answer = "".join(parts)
    return answer


async def main() -> None:
    a, b = [], []
    start = time.perf_counter()
    # 两个用户同时聊天，应该并发执行
    r1, r2 = await asyncio.gather(chat(a, "你好"), chat(b, "在吗"))
    cost = time.perf_counter() - start
    await chat(a, "我刚才说了什么？")
    print(r1, r2, f"耗时 {cost:.2f}s", f"a 有 {len(a)} 条消息")
    assert cost < 1.2, "两个请求没有并发起来"
    assert len(a) == 4, "历史里少了东西"
    print("✅ 全部修好了")


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await